In [14]:
%pip install python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

KEY: str = os.environ["KOREAN_DICT_KEY"]

print(KEY[:4] + "***")

7A7D***


In [24]:
import requests

def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    url: str = "https://opendict.korean.go.kr/api/search"
    
    params: dict = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word",
    }
    
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    
    return r.json()

search_word 함수는 우리말샘 API에 요청을 보낸 후 응답을 JSON으로 된 딕셔너리로 반환한다. 
검색어 q, 결과 개수 num, 시작 위치 start를 매개변수로 받아 다양한 검색에 재사용할 수 있도록 하였다.

In [26]:
import json

data: dict = search_word("김치")

print(json.dumps(data, ensure_ascii=False, indent=2)[:400])



{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


json.dumps를 사용하면 딕셔너리 형태의 응답 데이터를 알아보기 쉬운 문자열로 출력할 수 있다. 
ensure_ascii=False를 빼면 한글이 그대로 보이지 않고 유니코드 이스케이프 형식으로 보인다.

In [29]:
channel: dict = data["channel"]

total: int = int(channel["total"])
items: list[dict] = channel["item"]

n: int = len(items)

print(f"총 {total}건, 이 페이지 {n}건")

for item in items[:5]:
    word: str = item.get("word", "표제어 없음")
    pos: str = item.get("pos") or "품사 없음"

    senses = item.get("sense", [])

    if isinstance(senses, list) and len(senses) > 0:
        definition: str = senses[0].get("definition", "뜻풀이 없음")
    elif isinstance(senses, dict):
        definition: str = senses.get("definition", "뜻풀이 없음")
    else:
        definition: str = "뜻풀이 없음"

    print(f"{word} ({pos}) -> {definition[:40]}")

총 328건, 이 페이지 10건
김치 (품사 없음) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) -> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


응답 데이터의 channel에서 전체 결과 수 total과 실제 받은 항목 item을 꺼내 출력하였다. 
각 항목의 표제어, 품사, 뜻풀이를 출력할때 값이 없을 수 있으므로 get 메서드를 사용해 안전하게 접근하였다.

In [32]:
#q2
import time

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥",
]

all_items: list[dict] = []

for word in words:
    data: dict = search_word(word)
    
    channel: dict = data["channel"]
    total: int = int(channel["total"])
    items: list[dict] = channel["item"]
    
    print(f"{word}: {total}건")
    
    all_items.extend(items)
    
    time.sleep(0.3)

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


음식검색어 8개를 리스트에 저장한 후 for문으로 하나씩 search_word 함수에 전달하였다. 
각 검색어의 전체 결과 수는 응답 데이터의 data["channel"]["total"]에서 가져왔고,요청 사이에 time.sleep(0.3)을 넣어 서버에 부담을 최소화하였다.

In [33]:
from collections import Counter

pos_list: list[str] = []

for item in all_items:
    pos: str = item.get("pos") or "(미상)"
    pos_list.append(pos)

pos_count: Counter = Counter(pos_list)

for pos, count in pos_count.most_common(3):
    print(f"{pos}: {count}개")

(미상): 80개


8개 검색어에서 받은 모든 항목을 하나의 리스트로 합친 후 각 항목의 pos 값을 꺼내 품사 빈도를 계산하였다. 
pos가 없거나 빈 문자열인 경우에는 미상으로 처리했고, collections.Count의 most_common을 사용해 가장 많이 나온 품사 3개를 출력하였다.